# PdfExtractor — Google Colab GitHub Runner

Notebook ini hanya berfungsi sebagai runner. Source code pemrosesan tetap modular dan disimpan di repository GitHub; PDF dan hasil ekstraksi tetap disimpan di Google Drive.

Sebelum menjalankan, pastikan repository memiliki run_extraction.py, requirements.txt, folder PdfExtractor/, dan README.md di root (atau atur PROJECT_SUBDIR bila source berada di subfolder). Repository public paling sederhana untuk git clone tanpa token.

Alur: mount Drive → clone source GitHub → pilih runtime CPU/GPU → pilih engine VLM yang kompatibel → instal dependensi → validasi backend → ekstrak PDF → simpan database SQLite/CSV/JSONL dan aset.

Mode default adalah GPU. Untuk tier standard/advanced, engine auto memilih vLLM pada GPU NVIDIA, termasuk Tesla T4, dan llama-cpp pada CPU. Jalur GPU ini memerlukan wheel vLLM yang cocok dengan versi Python, CUDA, driver, dan GPU runtime. Pada laporan SIGSEGV sebelumnya, llama-cpp berhenti tepat setelah pemilihan engine; karena itu notebook tidak lagi memilih llama-cpp otomatis pada T4. Pilihan engine dapat dioverride di sel konfigurasi.


## 1. Konfigurasi — edit sel ini terlebih dahulu

Ganti URL GitHub, nama folder PDF, dan lokasi output sesuai Drive Anda. Jangan menaruh PDF atau hasil ekstraksi di repository GitHub.

In [ ]:
# ===== GANTI NILAI DI BAGIAN INI =====
GITHUB_REPO_URL = "https://github.com/zuudevs/PdfExtractor.git"
GITHUB_BRANCH = "main"
PROJECT_SUBDIR = ""  # Kosong jika run_extraction.py berada di root repo; selain itu isi subfolder, mis. "src"

# Runtime: "gpu" untuk NVIDIA CUDA di Colab; "cpu" untuk mode CPU.
DEVICE_MODE = "gpu"

# Tier: "flash", "basic", "standard", atau "advanced".
TIER = "standard"
OCR_MODE = "auto"  # "auto", "txt", atau "ocr"

# VLM engine: "auto", "vllm", atau "llama-cpp".
# Pada semua GPU NVIDIA Colab, "auto" memilih vllm; pada CPU, "auto" memilih llama-cpp.
# Pastikan instalasi full dan wheel vLLM kompatibel dengan runtime Colab.
VLM_ENGINE = "auto"
MODEL_SOURCE = "auto"  # "auto", "huggingface", atau "modelscope"

INPUT_DIR = "/content/drive/MyDrive/papers"
OUTPUT_DIR = "/content/drive/MyDrive/output/extracted_pdf"

# True pada run awal atau bila model belum tersedia di runtime saat ini.
# Cache model default Colab tidak persisten ketika runtime dibuang.
PREPARE_MODELS = True
RESUME_VALID_RESULTS = True
STOP_ON_ERROR = False


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Periksa pilihan GPU/CPU sebelum instalasi

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if DEVICE_MODE not in {"cpu", "gpu"}:
    raise ValueError("DEVICE_MODE harus 'cpu' atau 'gpu'.")
if TIER not in {"flash", "basic", "standard", "advanced"}:
    raise ValueError("TIER tidak valid.")
if VLM_ENGINE not in {"auto", "vllm", "llama-cpp"}:
    raise ValueError("VLM_ENGINE harus 'auto', 'vllm', atau 'llama-cpp'.")
if MODEL_SOURCE not in {"auto", "huggingface", "modelscope"}:
    raise ValueError("MODEL_SOURCE tidak valid.")

GPU_NAME = "CPU"
if DEVICE_MODE == "gpu":
    result = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            "GPU NVIDIA tidak terdeteksi. Di Colab, pilih Runtime > Change runtime type > GPU, "
            "lalu restart runtime. Jika ingin CPU, ubah DEVICE_MODE='cpu'."
        )
    name_result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        text=True,
        capture_output=True,
    )
    if name_result.returncode != 0 or not name_result.stdout.strip():
        raise RuntimeError("GPU terdeteksi tetapi nama model GPU gagal dibaca dari nvidia-smi.")
    GPU_NAME = name_result.stdout.strip().splitlines()[0]
    print(result.stdout[:5000])
else:
    print("Mode CPU dipilih. Tier standard/advanced mungkin memerlukan RAM dan waktu pemrosesan lebih besar.")

if TIER in {"standard", "advanced"}:
    if VLM_ENGINE == "auto":
        EFFECTIVE_VLM_ENGINE = "llama-cpp" if DEVICE_MODE == "cpu" else "vllm"
    else:
        EFFECTIVE_VLM_ENGINE = VLM_ENGINE
    if DEVICE_MODE == "cpu" and EFFECTIVE_VLM_ENGINE == "vllm":
        raise ValueError("vllm membutuhkan runtime GPU; gunakan VLM_ENGINE='llama-cpp'.")
else:
    EFFECTIVE_VLM_ENGINE = "not-used"

print(f"GPU            : {GPU_NAME}")
print(f"Mode / tier    : {DEVICE_MODE.upper()} / {TIER}")
print(f"VLM engine     : {EFFECTIVE_VLM_ENGINE}")
if DEVICE_MODE == "gpu" and EFFECTIVE_VLM_ENGINE == "vllm":
    print(
        "GPU VLM path selected. If vLLM fails its import/preflight, do not start a standard-tier batch; "
        "fix the wheel/runtime combination or use tier='basic' for a diagnostic run without VLM."
    )

input_path = Path(INPUT_DIR).expanduser()
output_path = Path(OUTPUT_DIR).expanduser()
if not input_path.exists() or not input_path.is_dir():
    raise FileNotFoundError(f"Folder PDF tidak ditemukan: {input_path}. Perbaiki INPUT_DIR.")
input_resolved = input_path.resolve()
output_resolved = output_path.resolve()
if input_resolved == output_resolved or input_resolved in output_resolved.parents:
    raise ValueError("OUTPUT_DIR harus terpisah dan tidak boleh berada di dalam INPUT_DIR.")

pdf_files = sorted(input_path.rglob("*.pdf"))
pdf_files += sorted(p for p in input_path.rglob("*.PDF") if p not in pdf_files)
if not pdf_files:
    raise FileNotFoundError(f"Tidak ditemukan file PDF di {input_path} atau subfoldernya.")
print(f"Jumlah PDF ditemukan: {len(pdf_files)}")
print(f"Input : {input_path}")
print(f"Output: {output_path}")


## 4. Clone atau perbarui source dari GitHub

In [ ]:
import shutil

if not GITHUB_REPO_URL.startswith("https://github.com/") or "YOUR_USERNAME" in GITHUB_REPO_URL:
    raise ValueError(
        "Isi GITHUB_REPO_URL dengan URL repository Anda, mis. https://github.com/nama/pdf-extractor.git"
    )

REPO_DIR = Path("/content/PdfExtractor")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone_cmd = ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(REPO_DIR)]
print("Menjalankan:", " ".join(clone_cmd))
subprocess.run(clone_cmd, check=True)

PROJECT_DIR = (REPO_DIR / PROJECT_SUBDIR).resolve() if PROJECT_SUBDIR else REPO_DIR.resolve()
if not PROJECT_DIR.is_relative_to(REPO_DIR.resolve()):
    raise ValueError("PROJECT_SUBDIR harus berada di dalam repository hasil clone.")

required_paths = [PROJECT_DIR / "run_extraction.py", PROJECT_DIR / "requirements.txt", PROJECT_DIR / "PdfExtractor"]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Struktur repository tidak sesuai. File/folder ini tidak ditemukan:\n- " + "\n- ".join(missing) +
        "\nUnggah isi project ke root repo, atau atur PROJECT_SUBDIR dengan benar."
    )
print("Source siap:", PROJECT_DIR)
print("File entry point:", PROJECT_DIR / "run_extraction.py")

## 5. Instal dependensi sesuai runtime

In [ ]:
# Base requirements memasang MinerU 4.x dan pypdf.
# GPU mode memakai extra [full] hanya saat VLM vLLM dipilih.
if DEVICE_MODE == "gpu":
    extra = "mineru[full]>=4.0,<5" if EFFECTIVE_VLM_ENGINE == "vllm" else "mineru[torch]>=4.0,<5"
    install_cmd = [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt"), extra]
else:
    install_cmd = [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")]

print("Perintah instalasi:", " ".join(install_cmd))
subprocess.run(install_cmd, check=True)
print("Instalasi selesai. Bila Colab meminta restart runtime, restart lalu jalankan ulang notebook dari awal.")


## 6. Validasi backend aktual sebelum memproses PDF

In [ ]:
import json
import inspect

# Samakan backend di pemeriksaan awal dengan yang akan dipakai CLI.
validation_env = os.environ.copy()
needs_vlm = TIER in {"standard", "advanced"}
if DEVICE_MODE == "cpu":
    validation_env["MINERU_MODEL_SMALL_BACKEND"] = "onnx"
    validation_env["MINERU_TABLE_DEVICE"] = "cpu"
    validation_env["CUDA_VISIBLE_DEVICES"] = "-1"
else:
    if validation_env.get("CUDA_VISIBLE_DEVICES") == "-1":
        validation_env.pop("CUDA_VISIBLE_DEVICES", None)
    validation_env["MINERU_MODEL_SMALL_BACKEND"] = "torch"
    validation_env["MINERU_TABLE_DEVICE"] = "cuda"

if needs_vlm:
    validation_env["MINERU_MODEL_VLM_ENGINE"] = EFFECTIVE_VLM_ENGINE
else:
    validation_env.pop("MINERU_MODEL_VLM_ENGINE", None)

if DEVICE_MODE == "gpu":
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError(
            "PyTorch tidak mendeteksi CUDA. Gunakan runtime GPU NVIDIA yang kompatibel "
            "atau ubah DEVICE_MODE='cpu'."
        )
    print("CUDA runtime (PyTorch):", torch.version.cuda)
    print("GPU PyTorch:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))
    if EFFECTIVE_VLM_ENGINE == "vllm":
        try:
            import vllm
            print("vLLM:", getattr(vllm, "__version__", "version tidak tersedia"))
        except Exception as exc:
            config_signature = "tidak tersedia"
            config_module_path = "tidak tersedia"
            try:
                from torch.utils._config_module import Config as TorchConfig
                config_signature = str(inspect.signature(TorchConfig))
                import torch.utils._config_module as config_module
                config_module_path = str(config_module.__file__)
            except Exception as diagnostic_exc:
                config_signature = f"diagnostik Config juga gagal: {type(diagnostic_exc).__name__}: {diagnostic_exc}"

            is_config_api_mismatch = (
                isinstance(exc, TypeError)
                and "deprecated" in str(exc)
                and "Config" in str(exc)
            )
            if is_config_api_mismatch:
                torch_release = torch.__version__.split("+", 1)[0]
                cuda_tag = (
                    "cu" + torch.version.cuda.replace(".", "")
                    if torch.version.cuda
                    else "cpu"
                )
                torch_index = f"https://download.pytorch.org/whl/{cuda_tag}"
                reinstall_command = (
                    f"{sys.executable} -m pip install --force-reinstall --no-cache-dir "
                    f"--no-deps --index-url {torch_index} 'torch=={torch_release}'"
                )
                raise RuntimeError(
                    "PyTorch dan modul internalnya tampak tidak sinkron: vLLM memicu "
                    "TypeError pada torch._dynamo.config saat Config(..., deprecated=...). "
                    "CUDA terdeteksi, tetapi preflight ini gagal sebelum MinerU berjalan. "
                    "Jangan lanjutkan ekstraksi Standard. Pada sel kode baru, jalankan perintah "
                    "di bawah untuk memasang ulang wheel PyTorch yang sama dengan versi/CUDA saat ini, "
                    "lalu pilih Runtime > Restart session dan jalankan notebook dari awal. "
                    "Perintah ini tidak mengubah versi dependensi lain.\n\n"
                    f"!{reinstall_command}\n\n"
                    f"Config signature: {config_signature}\n"
                    f"Config module: {config_module_path}\n"
                    f"Detail awal: {type(exc).__name__}: {exc}"
                ) from exc

            raise RuntimeError(
                "vLLM gagal diimpor sebelum ekstraksi. Periksa kecocokan wheel vLLM, "
                "PyTorch, Python, dan CUDA. Jangan jalankan ekstraksi Standard sebelum "
                "preflight berhasil. Untuk menguji jalur tanpa VLM, gunakan TIER='basic'. "
                f"Config signature: {config_signature}; module: {config_module_path}; "
                f"detail: {type(exc).__name__}: {exc}"
            ) from exc

print("Konfigurasi backend yang akan digunakan:")
for key in ["MINERU_MODEL_SMALL_BACKEND", "MINERU_MODEL_VLM_ENGINE", "MINERU_TABLE_DEVICE", "CUDA_VISIBLE_DEVICES"]:
    print(f"  {key} = {validation_env.get(key, '(unset)')}")

show_result = subprocess.run(
    ["mineru-kit", "models", "show"],
    cwd=str(PROJECT_DIR), env=validation_env, text=True, capture_output=True
)
print(show_result.stdout)
if show_result.stderr:
    print(show_result.stderr[-4000:])
if show_result.returncode != 0:
    raise RuntimeError("mineru-kit models show gagal. Periksa log di atas sebelum melanjutkan.")
print("Pemeriksaan backend selesai. Model akan diverifikasi ulang jika PREPARE_MODELS=True.")


## 7. Jalankan ekstraksi batch

In [ ]:
run_cmd = [
    sys.executable,
    str(PROJECT_DIR / "run_extraction.py"),
    "--input-dir", str(input_path),
    "--output-dir", str(output_path),
    "--device", DEVICE_MODE,
    "--tier", TIER,
    "--ocr-mode", OCR_MODE,
    "--model-source", MODEL_SOURCE,
]
if TIER in {"standard", "advanced"}:
    run_cmd.extend(["--vlm-engine", EFFECTIVE_VLM_ENGINE])
if PREPARE_MODELS:
    run_cmd.append("--prepare-models")
if not RESUME_VALID_RESULTS:
    run_cmd.append("--no-resume")
if STOP_ON_ERROR:
    run_cmd.append("--stop-on-error")

print("Perintah:", " ".join(run_cmd), flush=True)
print("Mulai pemrosesan. Mode standard/advanced dapat mengunduh model besar pada penggunaan pertama.", flush=True)
run_result = subprocess.run(run_cmd, cwd=str(PROJECT_DIR), env=os.environ.copy())
if run_result.returncode != 0:
    import signal
    from datetime import datetime, timezone

    run_error_path = output_path / "_RUN_ERROR.json"
    run_error_path.parent.mkdir(parents=True, exist_ok=True)
    signal_number = -run_result.returncode if run_result.returncode < 0 else None
    try:
        signal_name = signal.Signals(signal_number).name if signal_number is not None else None
    except ValueError:
        signal_name = f"SIG{signal_number}" if signal_number is not None else None
    report = {
        "failed_at_utc": datetime.now(timezone.utc).isoformat(),
        "command": run_cmd,
        "exit_code": run_result.returncode,
        "signal": signal_name,
        "device_mode": DEVICE_MODE,
        "gpu_name": GPU_NAME,
        "tier": TIER,
        "vlm_engine": EFFECTIVE_VLM_ENGINE,
        "python_executable": sys.executable,
        "python_version": sys.version,
        "working_directory": str(PROJECT_DIR),
        "diagnosis": (
            "Proses utama mati akibat signal native; Python tidak sempat menjalankan exception handler."
            if signal_number is not None else
            "Pipeline keluar dengan error nonzero sebelum menyelesaikan batch."
        ),
        "next_step": (
            "Periksa documents/*/_ERROR.json untuk crash saat parsing per PDF. Jika file itu tidak ada, "
            "crash terjadi di luar worker (misalnya setup/download model); uji tier='basic' terlebih dahulu "
            "untuk mengisolasi jalur VLM standard."
        ),
    }
    run_error_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    if signal_name:
        raise RuntimeError(
            f"Pipeline mati oleh {signal_name} (exit code {run_result.returncode}). "
            f"Diagnosis utama tersimpan di {run_error_path}. "
            "Jika crash terjadi saat parsing, periksa documents/*/_ERROR.json dan _WORKER.log."
        )
    raise RuntimeError(
        f"Pipeline keluar dengan exit code {run_result.returncode}. "
        f"Ringkasan diagnosis tersimpan di {run_error_path}."
    )
print("Proses ekstraksi selesai.")

## 8. Periksa hasil dan audit

In [ ]:
import csv

DB_DIR = output_path / "database"
expected_outputs = [
    DB_DIR / "literature_database.sqlite",
    DB_DIR / "literature_documents.csv",
    DB_DIR / "literature_tables.csv",
    DB_DIR / "literature_figures.csv",
    DB_DIR / "literature_text_blocks.csv",
    DB_DIR / "literature_elements.jsonl",
    DB_DIR / "literature_extraction_audit.csv",
    DB_DIR / "literature_image_inventory.csv",
]
print("Lokasi output:", output_path)
for item in expected_outputs:
    print(("[OK] " if item.exists() else "[BELUM ADA] ") + str(item))

audit_file = DB_DIR / "literature_extraction_audit.csv"
if audit_file.exists():
    with audit_file.open("r", encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    print(f"\\nBaris audit: {len(rows)}")
    if rows:
        print("Kolom audit:", list(rows[0].keys()))
        for row in rows[:10]:
            print({k: row.get(k) for k in list(row)[:6]})

print("\\nDatabase dan seluruh aset disimpan di Google Drive, bukan pada repository GitHub.")

## Catatan operasional

- Notebook mengharapkan repository GitHub memiliki run_extraction.py, requirements.txt, dan folder PdfExtractor/ di root (atau atur PROJECT_SUBDIR).
- DEVICE_MODE="gpu" ditujukan untuk GPU NVIDIA CUDA di Linux/Colab, bukan Radeon AMD pada Windows. Pada seluruh GPU NVIDIA, VLM_ENGINE="auto" memilih vLLM untuk tier standard/advanced, sementara small models memakai Torch dan model tabel dikonfigurasi CUDA. Pilihan ini memerlukan wheel vLLM yang kompatibel dengan runtime. Log Colab yang dilaporkan menunjukkan SIGSEGV setelah MinerU memilih llama-cpp, sehingga engine tersebut tidak lagi dipilih otomatis pada T4.
- Bila memilih VLM_ENGINE="vllm", notebook mencoba mengimpor vLLM sebelum ekstraksi. Jika import gagal, jangan lanjutkan ekstraksi Standard; periksa detail error dan sesuaikan wheel/runtime. Untuk menguji jalur ekstraksi tanpa VLM, ubah TIER="basic". Jangan otomatis beralih ke llama-cpp karena log Colab yang dilaporkan menunjukkan native SIGSEGV pada engine tersebut.
- PREPARE_MODELS=True mengunduh dan memverifikasi model untuk kombinasi backend yang dipilih. Cache model Colab umumnya hilang ketika runtime dibuang, sehingga pengunduhan bisa terulang pada sesi baru.
- Database, Markdown, JSON, CSV/JSONL, audit, dan aset gambar ditulis ke OUTPUT_DIR di Drive. Periksa literature_extraction_audit.csv dan bandingkan sampel hasil dengan PDF asli sebelum menggunakan hasil untuk analisis ilmiah.
- Dokumentasi resmi: https://opendatalab.github.io/MinerU/usage/tiers/, https://opendatalab.github.io/MinerU/quick_start/extension_modules/, https://opendatalab.github.io/MinerU/usage/model_source/.
